In [4]:
import collections
import copy
import math
import random
import re

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("sdadas/polish-gpt2-medium")
model = (
    AutoModelForCausalLM.from_pretrained("sdadas/polish-gpt2-medium")
    .half()
    .to("cuda")
    .eval()
)

Loading weights: 100%|██████████| 292/292 [00:01<00:00, 174.64it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: sdadas/polish-gpt2-medium
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
def stems(text):
    return {t.lower()[:5] for t in re.findall(r"\w+", text)}


docs = collections.defaultdict(set)
with open("plwiktionary_for_riddles.txt", encoding="utf-8") as f:
    for line in f:
        word, definition = line.split(";;", 1)
        docs[word.strip()] |= stems(definition)

with open("zagadki_do_testow_clean.txt", encoding="utf-8") as f:
    riddles = [tuple(p.strip() for p in line.split(";;")) for line in f]

random.seed(0)
torch.manual_seed(0)
random.shuffle(riddles)
test = riddles[50:200]

## Zadanie 2

Słownik to trie po tokenach `" słowo"`. W węźle wolno wybrać tylko token-dziecko, a jeśli węzeł kończy słowo, także **koniec słowa** o masie tokenów, które nie kontynuują wyrazu (to token nadmiarowy).

Żeby było szybko, `n` próbek rozdzielamy między dzieci węzła, więc każdy prefiks liczymy raz, a wszystkie prefiksy tej samej długości idą na GPU jednym batchem. Prompt liczymy tylko raz, dzięki `past_key_values`.

In [ ]:
boundary = torch.tensor(
    [i for i in range(len(tokenizer)) if not tokenizer.decode([i])[:1].isalpha()]
)


def build_trie(words):
    root = [{}, None]
    for word in words:
        node = root
        for token in tokenizer.encode(" " + word, add_special_tokens=False):
            node = node[0].setdefault(token, [{}, None])
        node[1] = word
    return root


@torch.no_grad()
def sample_words(prompt, n, trie, batch_size=128):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    out = model(input_ids=ids, use_cache=True)
    cache, logps = out.past_key_values, out.logits[:, -1].float().log_softmax(-1).cpu()
    frontier, counts = [(trie, n, [])], collections.Counter()
    while frontier:
        next_frontier = []
        for ((children, word), count, prefix), logp in zip(frontier, logps):
            options = logp[list(children)]
            if word:
                options = torch.cat(
                    [options, logp[boundary].logsumexp(0, keepdim=True)]
                )
            drawn = (
                options.softmax(0)
                .multinomial(count, replacement=True)
                .bincount(minlength=len(options))
                .tolist()
            )
            next_frontier += [
                (children[t], k, prefix + [t]) for t, k in zip(children, drawn) if k
            ]
            if word and drawn[-1]:
                counts[word] += drawn[-1]
        frontier = next_frontier
        logps = []
        for i in range(0, len(frontier), batch_size):
            batch = torch.tensor(
                [prefix for *_, prefix in frontier[i : i + batch_size]], device="cuda"
            )
            past = copy.deepcopy(cache)
            past.batch_repeat_interleave(len(batch))
            logps += (
                model(input_ids=batch, past_key_values=past)
                .logits[:, -1]
                .float()
                .log_softmax(-1)
                .cpu()
            )
    return counts


def make_prompt(riddle):
    return f"{riddle} Jedno słowo, które to opisuje, to:"


def report(name, rankings):
    ranks = [r.index(a) + 1 if a in r else None for (a, _), r in zip(test, rankings)]
    print(
        f"{name:<20} dokładność {ranks.count(1) / len(ranks):6.1%}, MRR {sum(1 / r for r in ranks if r) / len(ranks):.3f}"
    )


TRIE = build_trie(docs)
for n in [1, 20]:
    samples = [sample_words(make_prompt(riddle), n, TRIE) for _, riddle in test]
    assert all(w in docs for counts in samples for w in counts)
    report(
        f"LLM, n={n}", [[w for w, _ in counts.most_common(20)] for counts in samples]
    )

## Zadanie 3

TF-IDF po definicjach (rdzeń to 5 pierwszych liter) wybiera 100 kandydatów, a model generuje już tylko spośród nich. Końcowy wynik słowa to `log(1 + liczba wygenerowań) + α · TF-IDF`. Wagę `α = 1` dobrałem wcześniej na 50 zagadkach spoza zbioru testowego (`riddles[:50]`).

In [ ]:
doc_freq = collections.Counter(stem for doc in docs.values() for stem in doc)


def tfidf(riddle):
    query = stems(riddle)
    scores = {
        w: sum(math.log(len(docs) / doc_freq[s]) for s in doc & query)
        / math.sqrt(len(doc))
        for w, doc in docs.items()
    }
    return dict(sorted(scores.items(), key=lambda x: -x[1])[:100])


def rank(candidates, counts, alpha):
    return sorted(
        candidates,
        key=lambda w: math.log1p(counts[w]) + alpha * candidates[w],
        reverse=True,
    )[:20]


candidates = [tfidf(riddle) for _, riddle in test]
samples = [
    sample_words(make_prompt(riddle), 20, build_trie(c))
    for (_, riddle), c in zip(test, candidates)
]

for name, alpha in [
    ("LLM na kandydatach", 0),
    ("samo TF-IDF", 1000),
    ("LLM + TF-IDF", 1),
]:
    report(name, [rank(c, counts, alpha) for c, counts in zip(candidates, samples)])